# Silver 3: convert the pages to Markdown

Task `silver_convert` of the job `ingest_financebench`. One `ai_query` call per **page**. For every PDF in `silver_parsed`, the pages that are not in `silver_pages` yet (and have attempts left) are converted. Each result is checked: an error or an empty answer is logged as a failed page in `silver_log`, and anything else is stored in `silver_pages`.

The final documents are put together by the view `silver_documents` (see [tables.py](./tables.py)). It orders each PDF's pages, adds `<!-- page N -->` before each one and `---` between them, and shows a PDF only when **all** its pages are converted.

**Why one page per call**

- **The output limit.** The model endpoint returns at most 10,000 tokens per call. A whole filing in one call was silently cut off (a 150-page filing came back at about 43,000 characters). A page always fits.
- **Nothing to repair.** The model only converts the text of one page. It is told not to write page markers or separators, and `clean_page` removes any it writes anyway (it sometimes mistook the printed page number, such as "17", for a marker). The real marker and separator come from the view. In an earlier version the model wrote them, 151 of 359 documents had repeated or missing markers, and the repair code was most of the notebook.
- **Small retries.** A failed page costs one call to retry, not a whole document.

The page number is the real PDF page number (the parser's page id plus 1). Total tokens are about the same as converting several pages per call: only the short prompt repeats for every page.

**Budget and save points.** `max_pages_per_run` limits the calls of one run (`0` = none). Pages are converted in chunks of `pages_per_chunk`, and each chunk is saved before the next one, so a crash loses at most one chunk. A chunk that stores nothing stops the run.

The answers are collected and written once per chunk, so each paid `ai_query` call runs only once.

In [ ]:
from pyspark.sql import functions as F

from checks import clean_page
from tables import Names, create_tables, insert_new

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("endpoint", "databricks-qwen3-next-80b-a3b-instruct")
dbutils.widgets.text("max_attempts", "3")
dbutils.widgets.text("max_pages_per_run", "1000")
dbutils.widgets.text("pages_per_chunk", "200")

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("schema"))
endpoint = dbutils.widgets.get("endpoint")
max_attempts = int(dbutils.widgets.get("max_attempts"))
max_pages_per_run = int(dbutils.widgets.get("max_pages_per_run"))
pages_per_chunk = int(dbutils.widgets.get("pages_per_chunk"))

create_tables(spark, n)

In [ ]:
PROMPT = """Convert the parsed elements of ONE page of a financial filing (PDF) into clean Markdown.

The input is a JSON array in reading order. Each element has a "type" (title, section_header, text, table, list_item, caption, footnote, figure, page_header, page_footer, page_number, ...) and its "content" (tables are HTML).

Rules:
- Keep ALL content exactly as written. Do not summarize, reword, reorder or add anything. Copy numbers, names and punctuation exactly.
- A "title" becomes "# Title". A "section_header" becomes "## Heading" (use "###" for clearly nested sections). Never turn normal text into a heading.
- Keep plain text as paragraphs, and lists as Markdown lists.
- Convert tables to Markdown pipe tables with a header row and a "| --- |" row. Keep every row and column.
- Keep page headers, page footers and page numbers as italic lines (*like this*).
- Output ONLY the Markdown of this page. Do not write page markers, "---" separators, code fences or commentary.

Elements:
"""

In [ ]:
def pages_to_convert(limit):
    """Pages of parsed PDFs that are not converted yet and have attempts left."""
    pages = spark.sql(f"""
    SELECT d.path, coalesce(e.value:bbox[0].page_id::INT, 0) + 1 AS page_no,
           e.pos, e.value:type::STRING AS type, e.value:content::STRING AS content
    FROM {n.parsed} d
    LEFT JOIN (SELECT path, count(*) AS n FROM {n.pages} GROUP BY path) g ON g.path = d.path
    , LATERAL variant_explode(d.parsed:document:elements) e
    WHERE coalesce(g.n, 0) < d.pages
    """).groupBy("path", "page_no").agg(
        F.to_json(F.transform(F.array_sort(F.collect_list(F.struct("pos", "type", "content"))),
                              lambda x: F.struct(x["type"], x["content"]))).alias("elements")
    )
    done = spark.table(n.pages).select("path", "page_no")
    given_up = (
        spark.table(n.log).filter("stage = 'convert'").groupBy("path", "page_no").count()
        .filter(f"count >= {max_attempts}").select("path", "page_no")
    )
    return pages.join(done, ["path", "page_no"], "left_anti").join(given_up, ["path", "page_no"], "left_anti").limit(limit)


def convert_chunk(limit):
    """Convert up to `limit` pages. Good pages go to silver_pages, bad ones to silver_log. Returns (stored, failed)."""
    rows = (
        pages_to_convert(limit)
        .withColumn("prompt", F.concat(F.lit(PROMPT), F.col("elements")))
        .repartition(32)  # run the calls in parallel
        .select("path", "page_no", F.expr(
            f"ai_query('{endpoint}', prompt, modelParameters => named_struct('temperature', 0.0, 'max_tokens', 10000), failOnError => false)"
        ).alias("r"))
        .collect()
    )
    attempts = {
        (r.path, r.page_no): r.n
        for r in spark.table(n.log).filter("stage = 'convert'").groupBy("path", "page_no").agg(F.max("attempt").alias("n")).collect()
    }
    good, bad = [], []
    for r in rows:
        text = clean_page(r.r.result)
        if r.r.errorMessage or not text:
            reason = (r.r.errorMessage or "empty answer")[:500]
            bad.append((r.path, "convert", r.page_no, attempts.get((r.path, r.page_no), 0) + 1, True, reason, None))
        else:
            good.append((r.path, r.page_no, text))
    if good:
        pages = spark.createDataFrame(good, "path string, page_no int, markdown string").withColumn("converted_at", F.current_timestamp())
        insert_new(spark, pages, n.pages, ["path", "page_no"])
    if bad:
        logged = spark.createDataFrame(
            bad, "path string, stage string, page_no int, attempt int, retryable boolean, reason string, pdf_pages int"
        ).withColumn("logged_at", F.current_timestamp())
        insert_new(spark, logged, n.log, ["path", "stage", "page_no", "attempt"])
    return len(good), len(bad)

In [ ]:
spent = 0
while spent < max_pages_per_run:
    stored, failed = convert_chunk(min(pages_per_chunk, max_pages_per_run - spent))
    spent += stored + failed
    print(f"{stored} pages stored, {failed} failed")
    if stored == 0:
        break
print(f"Pages converted in this run: {spent} of a budget of {max_pages_per_run}.")
print("Complete documents (silver_documents):", spark.table(n.documents).count())